# Hybrid Intelligent Network Intrusion Detection System (H-NIDS)
## Phase 1 Working ML Prototype — Faculty Evaluation Benchmark

> **Academic Stage:** 7th Semester Major Project  
> **Objective:** Demonstrate working supervised machine learning baselines (Random Forest & XGBoost) on authentic network intrusion traffic (CIC-IDS2017) with full evaluation metrics, confusion matrices, and unseen flow predictions before advancing to Phase 2 (TabTransformer + Autoencoder + Hybrid Decision Engine).

### Step 1: Ingest and Inspect the Dataset
We load the Canadian Institute for Cybersecurity benchmark (`CIC-IDS2017`), examine the flow attributes, and inspect nulls, infinite values, and label distribution.

In [1]:
import os
import sys
# Ensure project root is in sys.path for robust module imports
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
from src.data_loader import NetworkDataLoader

dataset_path = 'data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
loader = NetworkDataLoader(dataset_path)
df_raw = loader.load_data(sample_size=100000, random_state=42)
inspection = loader.inspect_dataset()

print(f"Total Flows Loaded: {len(df_raw):,}")
print(f"Total Features: {inspection['total_features']}")
print(f"Class Distribution: {inspection['class_distribution']}")
print(f"Infinite Values: {inspection['total_infinite_values']:,}")
print(f"Missing Values: {inspection['total_null_values']:,}")

[*] Loading network traffic dataset from: data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
[*] Sampling 100,000 records from 286,467 total flows (stratified)...
Total Flows Loaded: 100,000
Total Features: 78
Class Distribution: {'PortScan': {'count': 55479, 'percentage': 55.48}, 'BENIGN': {'count': 44521, 'percentage': 44.52}}
Infinite Values: 247
Missing Values: 3


### Step 2: Data Cleaning & Preprocessing (Strict Featurization Ordering)
1. Strip column whitespace.
2. Remove duplicate flows.
3. Replace division-by-zero infinities (`Flow Bytes/s`) with training medians.
4. Eliminate invariant zero-variance features.
5. **Stratified 80/20 Train/Test split BEFORE parameter fitting** to guarantee zero data leakage.
6. Apply `RobustScaler` on training split.

In [2]:
from src.preprocessor import NetworkDataPreprocessor
from src.visualize import NetworkVisualizer

preprocessor = NetworkDataPreprocessor(target_col=inspection['label_column'], scaler_type='robust')
df_clean, cleaning_stats = preprocessor.clean_raw_dataframe(df_raw)

# Visualize class distribution
visualizer = NetworkVisualizer('artifacts/figures')
clean_dist = {
    k: {'count': int(v), 'percentage': round(v / len(df_clean) * 100, 2)}
    for k, v in df_clean[inspection['label_column']].value_counts().items()
}
visualizer.plot_class_distribution(clean_dist)

# Leakage-free train/test split
X_train, X_test, y_train, y_test, meta = preprocessor.prepare_train_test_split(df_clean, test_size=0.20, random_state=42)
print(f"Training Set: {len(X_train):,} flows | Test Set: {len(X_test):,} flows")

[*] Removed 13,761 duplicate flows (13.76%).
[*] Replaced 237 infinite values with NaN for robust imputation.
[*] Saved class distribution plot to: artifacts/figures\class_distribution.png
[*] Encoded Target Classes: {0: 'BENIGN', 1: 'PortScan'}
[*] Dropping 10 constant/zero-variance columns: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
[*] Train set: 68,991 flows | Test set: 17,248 flows (test_size=20%)
Training Set: 68,991 flows | Test Set: 17,248 flows


### Step 3: Train Random Forest and XGBoost Models
- **Random Forest**: 100 estimators, max depth 20, class weighting.
- **XGBoost**: Histogram-based gradient boosted trees, max depth 6, learning rate 0.1.

In [3]:
from src.models import MLModelTrainer

trainer = MLModelTrainer(random_state=42)
rf_model, rf_time = trainer.train_random_forest(X_train, y_train, n_estimators=100, max_depth=20)
xgb_model, xgb_time = trainer.train_xgboost(X_train, y_train, n_estimators=100, max_depth=6, learning_rate=0.1)


[+] Training Random Forest Classifier (n_estimators=100, max_depth=20)...
[+] Random Forest training completed in 0.96 seconds.

[+] Training XGBoost Classifier (n_estimators=100, max_depth=6, lr=0.1)...
[+] XGBoost training completed in 0.76 seconds.


### Step 4: Model Evaluation on Holdout Test Set (17,248 Unseen Flows)
Calculate actual experimental metrics: Accuracy, Macro/Weighted Precision, Recall, F1-Score, False Positive Rate (FPR), False Negative Rate (FNR), Confusion Matrix, and Inference Latency.

In [4]:
from src.evaluate import ModelEvaluator

evaluator = ModelEvaluator(class_names=preprocessor.classes_)
rf_eval = evaluator.evaluate_model('Random Forest', rf_model, X_test, y_test, rf_time)
xgb_eval = evaluator.evaluate_model('XGBoost', xgb_model, X_test, y_test, xgb_time)


[*] Evaluating Random Forest on 17,248 holdout test samples...
    - Accuracy:  99.99%
    - Precision: 99.99% (macro) | 99.99% (weighted)
    - Recall:    99.99% (macro) | 99.99% (weighted)
    - F1-Score:  99.99% (macro) | 99.99% (weighted)
    - False Positive Rate (FPR): 0.0000%
    - False Negative Rate (FNR): 0.0235%
    - ROC-AUC:   0.9999
    - Latency:   2.49 µs/flow (401,042 flows/sec)

[*] Evaluating XGBoost on 17,248 holdout test samples...
    - Accuracy:  99.99%
    - Precision: 99.99% (macro) | 99.99% (weighted)
    - Recall:    99.99% (macro) | 99.99% (weighted)
    - F1-Score:  99.99% (macro) | 99.99% (weighted)
    - False Positive Rate (FPR): 0.0000%
    - False Negative Rate (FNR): 0.0117%
    - ROC-AUC:   1.0000
    - Latency:   0.40 µs/flow (2,517,295 flows/sec)


### Step 5: Confusion Matrices & Comparative Visualizations

In [5]:
import numpy as np

visualizer.plot_confusion_matrix(np.array(rf_eval['confusion_matrix']), preprocessor.classes_, 'Random Forest')
visualizer.plot_confusion_matrix(np.array(xgb_eval['confusion_matrix']), preprocessor.classes_, 'XGBoost')
visualizer.plot_model_comparison(rf_eval, xgb_eval)
visualizer.plot_feature_importances(preprocessor.feature_columns, rf_model, xgb_model, top_k=12)

[*] Saved confusion matrix plot to: artifacts/figures\confusion_matrix_random_forest.png
[*] Saved confusion matrix plot to: artifacts/figures\confusion_matrix_xgboost.png
[*] Saved model comparison plot to: artifacts/figures\model_comparison.png
[*] Saved feature importance plot to: artifacts/figures\feature_importance.png


### Step 6: Live Inference on Unseen Test Network Flows
Pass individual held-out test flows through both models to demonstrate live prediction accuracy and confidence scores.

In [6]:
from src.sample_inference import UnseenFlowPredictor

predictor = UnseenFlowPredictor(preprocessor, rf_model, xgb_model, preprocessor.classes_)
raw_test_df = df_clean.iloc[-len(y_test):].reset_index(drop=True)
unseen_preds = predictor.run_sample_predictions(X_test, y_test, raw_test_df, num_samples_per_class=4)


 LIVE INFERENCE DEMONSTRATION ON UNSEEN HELD-OUT TEST NETWORK FLOWS (8 SAMPLES)
Sample #01 | True: PortScan   | [MATCH]
   Key Features: {'Destination Port': 81.0, 'Flow Duration': 45.0, 'Total Fwd Packets': 1.0, 'Total Backward Packets': 1.0, 'Fwd Packet Length Mean': 2.0, 'Flow Bytes/s': 177777.78}
   -> Random Forest: PortScan   (Confidence: 100.00%)
   -> XGBoost:       PortScan   (Confidence: 100.00%)
-------------------------------------------------------------------------------------
Sample #02 | True: PortScan   | [MATCH]
   Key Features: {'Destination Port': 617.0, 'Flow Duration': 42.0, 'Total Fwd Packets': 1.0, 'Total Backward Packets': 1.0, 'Fwd Packet Length Mean': 2.0, 'Flow Bytes/s': 190476.19}
   -> Random Forest: PortScan   (Confidence: 100.00%)
   -> XGBoost:       PortScan   (Confidence: 100.00%)
-------------------------------------------------------------------------------------
Sample #03 | True: BENIGN     | [MATCH]
   Key Features: {'Destination Port': 3011.0, 

### Step 7: Random Forest vs. XGBoost Benchmark Summary Table

In [7]:
comp_df = pd.DataFrame([
    {'Metric': 'Accuracy', 'Random Forest': f"{rf_eval['accuracy']*100:.3f}%", 'XGBoost': f"{xgb_eval['accuracy']*100:.3f}%"},
    {'Metric': 'Precision (Macro)', 'Random Forest': f"{rf_eval['precision_macro']*100:.3f}%", 'XGBoost': f"{xgb_eval['precision_macro']*100:.3f}%"},
    {'Metric': 'Recall (Macro)', 'Random Forest': f"{rf_eval['recall_macro']*100:.3f}%", 'XGBoost': f"{xgb_eval['recall_macro']*100:.3f}%"},
    {'Metric': 'F1-Score (Macro)', 'Random Forest': f"{rf_eval['f1_score_macro']*100:.3f}%", 'XGBoost': f"{xgb_eval['f1_score_macro']*100:.3f}%"},
    {'Metric': 'False Positive Rate (FPR)', 'Random Forest': f"{rf_eval['false_positive_rate']*100:.4f}%", 'XGBoost': f"{xgb_eval['false_positive_rate']*100:.4f}%"},
    {'Metric': 'False Negative Rate (FNR)', 'Random Forest': f"{rf_eval['false_negative_rate']*100:.4f}%", 'XGBoost': f"{xgb_eval['false_negative_rate']*100:.4f}%"},
    {'Metric': 'Training Time', 'Random Forest': f"{rf_eval['training_time_seconds']:.2f} s", 'XGBoost': f"{xgb_eval['training_time_seconds']:.2f} s"},
    {'Metric': 'Inference Latency per Flow', 'Random Forest': f"{rf_eval['latency_microseconds_per_flow']:.2f} µs", 'XGBoost': f"{xgb_eval['latency_microseconds_per_flow']:.2f} µs"},
    {'Metric': 'Throughput (Flows/sec)', 'Random Forest': f"{rf_eval['throughput_flows_per_sec']:,.0f}", 'XGBoost': f"{xgb_eval['throughput_flows_per_sec']:,.0f}"}
])
display(comp_df)

                       Metric Random Forest    XGBoost
0                    Accuracy       99.988%    99.994%
1           Precision (Macro)       99.989%    99.994%
2              Recall (Macro)       99.988%    99.994%
3            F1-Score (Macro)       99.988%    99.994%
4   False Positive Rate (FPR)       0.0000%    0.0000%
5   False Negative Rate (FNR)       0.0230%    0.0120%
6               Training Time        0.96 s     0.76 s
7  Inference Latency per Flow       2.49 µs    0.40 µs
8      Throughput (Flows/sec)       401,042  2,517,295


### Step 8: Architecture Roadmap & Faculty Viva Statement

> **Suggested Explanation to Faculty:**  
> *“In the first phase, we implemented supervised machine learning for network intrusion detection using Random Forest and XGBoost. We are evaluating the models using precision, recall, F1-score and confusion matrices. The proposed next phase adds TabTransformer for attention-based classification and an Autoencoder for anomaly detection of held-out or suspicious attack behavior, followed by hybrid decision fusion and Explainable AI.”*

**Why Phase 2 is Required:**
1. **Supervised Limitations:** Supervised models assume a closed-world distribution; novel zero-day attacks are forced into known categories.
2. **TabTransformer:** Embeds tabular flow features through multi-head self-attention to capture complex inter-feature correlations.
3. **Deep Autoencoder:** Trained strictly on benign network traffic to learn baseline normal behavior; novel attacks produce high reconstruction error.
4. **Hybrid Decision Fusion:** Combines supervised attack classification probability with unsupervised anomaly score to classify flows as `Normal`, `Known Attack`, or `Suspicious/Held-Out`.
5. **Explainable AI (SHAP):** Generates flow-level attribution values for security analysts.